# Notebook 2: Climatology & Index Construction

This notebook analyses the seasonal climatology and long term trends of the study region, then constructs the compound Wind Hydro Drought Index (WHDI):

1. Monthly climatology and seasonal cycles
2. Long term trends for all variables
3. Standardisation of wind speed and runoff
4. WHDI construction (WHDI1, WHDI3, WHDI6)
5. Drought event cataloguing
6. Component correlation analysis

**Key question for this notebook:** Do wind and runoff anomalies co occur, justifying a compound index? If they're uncorrelated, compound droughts are random coincidences rather than a coherent phenomenon.

## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv(
    "../data/processed/combined_dataset.csv", index_col="date", parse_dates=True
)

print(f"Data loaded: {df.shape}")
print(f"Period: {df.index[0]} to {df.index[-1]}")
print(f"Variables: {list(df.columns)}")
print("\nMissing Values:")
print(df.isna().sum())
print("\nFirst few rows:")
df.head()

Data loaded: (564, 8)
Period: 1979-01-01 00:00:00 to 2025-12-01 00:00:00
Variables: ['wind_speed', 'temperature', 'precipitation', 'runoff', 'snowmelt', 'sam', 'oni', 'iod']

Missing Values:
wind_speed       0
temperature      0
precipitation    0
runoff           0
snowmelt         0
sam              0
oni              0
iod              8
dtype: int64

First few rows:


,wind_speed,temperature,precipitation,runoff,snowmelt,sam,oni,iod
date,,,,,,,,
1979-01-01,3.591189,20.300608,33.127508,4.771823,0.007942,0.74,0.03,0.317
1979-02-01,4.052137,19.015506,13.530812,2.303862,0.008822,-0.90,0.07,-0.158
1979-03-01,2.668088,15.476255,31.482286,2.530781,0.201180,1.51,0.20,-0.034
1979-04-01,2.939223,12.290993,11.758993,1.787631,0.171488,-0.49,0.28,-0.207
1979-05-01,4.399244,7.483713,74.614402,12.958914,18.369194,1.54,0.23,-0.444


## Monthly Climatology

In [9]:
# Computing long term mean and standard deviation of each month
climatology = df.groupby(df.index.month).agg(["mean", "std"])

# Flatten the multi level column names
climatology.columns = [f"{var}_{stat}" for var, stat in climatology.columns]

climatology.to_csv("../results/tables/monthly_climatology.csv")

print("Monthly Climatology:")
climatology.round(3)

Monthly Climatology:


,wind_speed_mean,wind_speed_std,temperature_mean,temperature_std,precipitation_mean,precipitation_std,runoff_mean,runoff_std,snowmelt_mean,snowmelt_std,sam_mean,sam_std,oni_mean,oni_std,iod_mean,iod_std
date,,,,,,,,,,,,,,,,
1,4.313,0.897,19.291,0.946,28.048,11.665,5.633,1.702,0.393,0.384,0.940,1.700,0.045,1.074,-0.013,0.249
2,3.531,1.007,18.362,1.079,29.591,13.327,3.772,1.338,0.376,0.611,0.198,1.942,0.047,0.925,-0.032,0.232
3,3.464,0.900,15.474,0.952,37.443,13.961,4.250,1.597,0.845,0.870,0.247,1.663,0.055,0.744,-0.039,0.265
4,3.928,0.958,11.148,1.090,44.142,20.254,6.737,3.776,4.395,3.654,0.554,1.638,0.060,0.607,-0.019,0.262
5,4.053,1.002,7.325,1.088,55.093,24.262,13.327,7.656,11.872,7.465,0.320,1.510,0.063,0.557,-0.037,0.306
6,4.505,1.029,4.405,1.300,62.702,20.919,18.333,6.420,19.415,7.044,0.390,1.786,0.055,0.575,-0.046,0.346
7,4.141,1.050,3.742,1.265,53.924,21.125,16.671,6.009,17.787,7.015,0.329,1.622,0.046,0.642,-0.056,0.367
8,4.111,0.844,5.631,0.916,51.338,15.743,16.614,6.443,20.211,6.783,-0.054,1.708,0.037,0.743,-0.052,0.412
9,3.683,0.987,8.096,0.882,41.837,16.131,17.866,4.574,20.886,5.733,-0.078,1.678,0.030,0.869,-0.046,0.426
